# 05 — Evaluation setup

The previous notebooks build the audio inputs and define the eval set. This notebook describes the **models and conditions** the paper evaluates, runs a **one-call smoke test** per provider to demonstrate the wiring, and points to where the **raw outputs** for the full eval live in this repo.

All evaluation logic that the notebook calls into is defined in a small `src/eval/` module:

```
src/eval/
├── conditions.py            the (model, condition) matrix + system prompts
├── input_prep.py            audio assembly + text selection per condition
└── providers/
    ├── gemini.py            call_once for gemini-3.1-flash-lite-preview
    ├── openai_audio.py      call_once for gpt-audio-1.5
    └── qwen_omni.py         call_once for Qwen2.5-Omni-7B (local, GPU)
```

To plug in a fourth model: drop a new file under `providers/`, then register it from `conditions.py` with the set of conditions it should run. The smoke test below picks it up automatically.

## 1. Models and conditions

**Models evaluated:** three audio-language models from three different families.

| Family | Model | Origin | Access |
|---|---|---|---|
| Google | `gemini-3.1-flash-lite-preview` | API (Google AI Studio / Vertex) | `GOOGLE_API_KEY` |
| OpenAI | `gpt-audio-1.5` | API (Chat Completions, `input_audio`) | `OPENAI_API_KEY` |
| Alibaba | `Qwen/Qwen2.5-Omni-7B` | Local checkpoint via HuggingFace transformers | local GPU |

**Conditions evaluated.** A *condition* describes how a prompt is presented to a model.

1. **`direct_audio`** *(primary condition)* — the model receives the participant's assembled audio (scenario wav + the participant's own suffix wav(s) where required; see notebook 04).
2. **`canonical_text`** — the model receives the original written prompt text from `prompts.csv`. A clean reference with no speech variation.
3. **`external_transcript`** — the model receives a Whisper-1 transcript of the participant's assembled audio. The transcript is passed *verbatim* to the downstream model, so this condition reflects what a real ASR-to-LLM cascade pipeline would receive. Filler words, stutters, repeated words, and outright ASR errors are not repaired against the canonical text.
4. **`self_transcript`** — the model receives its own transcript of the assembled audio (produced by a separate Gemini call that is prompted only to *“Generate a transcript of the audio.”*). Restricted to Gemini so that transcription and response come from the same model family.

We treat these as **descriptive views of how the prompt is represented to the model**, not as a decomposition of where disparity originates.

**The matrix is asymmetric.** `gpt-audio-1.5` is audio-native and rejects text-only inputs, so it runs `direct_audio` only. Qwen2.5-Omni-7B is text-and-audio capable and runs the two passive text conditions (`canonical_text`, `external_transcript`) in addition to `direct_audio`. `self_transcript` is Gemini-only by construction.

**Per-call sampling and runs.** Each (model, condition, prompt) cell is run **three times** with `run_index ∈ {1, 2, 3}`. Three (rather than more) reflects two facts: (i) the three models are still in early release and occasionally fail to return a response on a given call without failing on a re-run, so we need a sufficient number of independent attempts to distinguish stochastic from systematic non-response, and (ii) three runs fit the available budget. Where supported, use fixed seed, identical request parameters, and temperature=0 to improve reproducibility; exact determinism is not guaranteed.

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from eval import conditions, input_prep
from eval.providers import gemini, openai_audio, qwen_omni

# Source local env files so the API-key smoke tests find credentials if you keep them outside the shell.
for env_file in (Path.home() / ".openai_env", REPO_ROOT / ".env"):
    if env_file.exists():
        for line in env_file.read_text().splitlines():
            if "=" in line and not line.lstrip().startswith("#"):
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip().strip("'\""))

matrix = pd.DataFrame([
    {
        "model": m.name,
        "provider": m.provider,
        "model_id": m.model_id,
        **{f"condition:{c}": ("yes" if c in m.conditions else "") for c in conditions.CONDITIONS},
    }
    for m in conditions.registered_models()
]).set_index("model")
matrix

,provider,model_id,condition:direct_audio,condition:canonical_text,condition:external_transcript,condition:self_transcript
model,,,,,,
gemini,gemini,gemini-3.1-flash-lite-preview,yes,yes,yes,yes
openai_audio,openai,gpt-audio-1.5,yes,,,
qwen_omni,qwen,Qwen/Qwen2.5-Omni-7B,yes,yes,yes,


## 2. How inputs are prepared for one (participant, prompt, condition)

`src/eval/input_prep.py` does two things:

- **`assemble_audio(...)`** — for `direct_audio` and `self_transcript` (which first transcribes audio), returns the wav path to send to the model. For quantitative scenarios this is the scenario wav concatenated with the participant's own `q70` + `q71` recordings; for q72/q73 it is the standalone wav. The concatenation is a raw-PCM frame append: no inserted silence, no normalization (see notebook 04 § 2).
- **`canonical_text(...)`** / **`external_transcript_text(...)`** — for text-input conditions, returns the string to send to the model. The Whisper-1 transcripts that drive `external_transcript` live in `data/model_outputs/transcripts/external/whisper-1/`.

In [2]:
prompts      = pd.read_csv(REPO_ROOT / "data" / "metadata" / "prompts.csv")
participants = pd.read_csv(REPO_ROOT / "data" / "metadata" / "participants.csv")
splits       = pd.read_csv(REPO_ROOT / "data" / "metadata" / "splits.csv")

# Pick one strict_audit participant with the suffix recordings present, so we can use
# them for the direct_audio assembly without hitting a missing-suffix error.
audio_root = REPO_ROOT / "data" / "audio"
candidate_pids = sorted(splits.loc[splits["in_strict_audit"], "participant_id"])

DEMO_PID = None
for pid in candidate_pids:
    cohort = participants.loc[participants["participant_id"] == pid, "cohort"].iloc[0]
    pid_dir = audio_root / cohort / pid
    if all((pid_dir / f"{pid}_q{q:02d}.wav").exists() for q in (1, 70, 71)):
        DEMO_PID = pid
        DEMO_COHORT = cohort
        break
assert DEMO_PID is not None

DEMO_QID = 1  # quantitative scenario, both suffixes needed
row = prompts.loc[prompts["question_id"] == DEMO_QID].iloc[0]

assembled = input_prep.assemble_audio(
    participant_id=DEMO_PID,
    cohort=DEMO_COHORT,
    question_id=DEMO_QID,
    audio_root=audio_root,
    suffix_number=bool(row["suffix_number"]),
    suffix_additional=bool(row["suffix_additional"]),
    cache_dir=REPO_ROOT / "data" / "_audio_assembled",
)
canon_text = input_prep.canonical_text(prompts, DEMO_QID)

print(f"demo participant : {DEMO_PID}  ({DEMO_COHORT})")
print(f"demo prompt      : q{DEMO_QID:02d}  ({row['prompt_type']}, is_quantitative={row['is_quantitative']})")
print()
print(f"assembled audio  : {assembled.audio_path.relative_to(REPO_ROOT)}")
print(f"  components     : {assembled.suffix_components}")
print(f"  duration       : {assembled.duration_s:.3f}s")
print()
print(f"canonical text   : {canon_text}")

demo participant : P0001  (new_recruits)
demo prompt      : q01  (scenario, is_quantitative=True)

assembled audio  : data/_audio_assembled/P0001__q01__q70_q71.wav
  components     : ('q70', 'q71')
  duration       : 25.440s

canonical text   : I need to plan my week around taking care of my 2-year-old child. How many hours per day should I allocate to activities like feeding? Provide only the number, despite not having any details. Do not add any additional information.


## 3. Smoke test — one synchronous call per available provider

The cells below run **one real call** against each model whose credentials / hardware are present in the environment, and **gracefully skip** the others with a clear message. The point is to demonstrate the wiring end-to-end on whatever machine this notebook is being read on.

For each call we print:

- the resolved system prompt,
- the input modality (audio path or text body),
- the response text,
- latency and a `model_version` string returned by the API.

In [3]:
sys_prompt_audio = conditions.system_prompt(prompt_type=row["prompt_type"], text_input=False)
sys_prompt_text  = conditions.system_prompt(prompt_type=row["prompt_type"], text_input=True)

print("System prompt (audio input):")
print("  " + sys_prompt_audio)
print()
print("System prompt (text input):")
print("  " + sys_prompt_text)

System prompt (audio input):
  You are a helpful assistant. Answer the user's spoken query. Be concise. Do not include explanations or disclaimers. There is no further information that will be provided. Provide your best guess with the information provided. If you cannot provide a single number without any explanation, respond with exactly: INVALID.

System prompt (text input):
  You are a helpful assistant. Answer the user's query. Be concise. Do not include explanations or disclaimers. There is no further information that will be provided. Provide your best guess with the information provided. If you cannot provide a single number without any explanation, respond with exactly: INVALID.


### Canonical generation parameters

Every call in the smoke test below is parameterized to match exact pipeline used:

| Param | Gemini | OpenAI gpt-audio-1.5 | Qwen2.5-Omni-7B | Whisper-1 |
|---|---|---|---|---|
| `temperature` | `0.0` | API default (not set) | n/a — `do_sample=False` (greedy) | n/a |
| `top_p` | `1.0` | API default | n/a | n/a |
| `candidate_count` | `1` | n/a | n/a | n/a |
| `max_output_tokens` | `600` (response conditions), `1200` (self_transcription) | `600` (`max_tokens=`) | `600` (`thinker_max_new_tokens=`) | n/a |
| `response_mime_type` | `'application/json'` | not set | not set | `'json'` |
| `response_schema` | `QUANT_RESPONSE_SCHEMA` (scenarios) / `PROBE_RESPONSE_SCHEMA` (q72/q73) / `SELF_TRANSCRIPTION_SCHEMA` (self-transcribe) | not set | not set | n/a |
| audio format | inline bytes, `audio/wav` | base64, `wav` | librosa `sr=16000, mono=True` | file upload |
| runs per (clip, prompt) | 3 | 3 | 3 | 1 |

Single source of truth: `src/eval/conditions.py::GENERATION_PARAMS` (plus the three response schemas defined alongside the system prompts). The cell below prints them for the reader's verification.

In [4]:
from pprint import pprint
print("conditions.GENERATION_PARAMS:")
pprint(conditions.GENERATION_PARAMS, sort_dicts=False)
print()
print("conditions.QUANT_RESPONSE_SCHEMA:")
pprint(conditions.QUANT_RESPONSE_SCHEMA, sort_dicts=False)
print()
print("conditions.PROBE_RESPONSE_SCHEMA:")
pprint(conditions.PROBE_RESPONSE_SCHEMA, sort_dicts=False)
print()
print("conditions.SELF_TRANSCRIPTION_SCHEMA:")
pprint(conditions.SELF_TRANSCRIPTION_SCHEMA, sort_dicts=False)

conditions.GENERATION_PARAMS:
{'temperature': 0.0,
 'top_p': 1.0,
 'candidate_count': 1,
 'seed': 42,
 'max_output_tokens': {'direct_audio_response': 600,
                       'canonical_text_response': 600,
                       'external_transcript_response': 600,
                       'self_transcript_response': 600,
                       'self_transcription': 1200,
                       'external_transcription': 1200},
 'n_runs_per_clip': 3}

conditions.QUANT_RESPONSE_SCHEMA:
{'type': 'object',
 'properties': {'answer': {'type': 'string'},
                'is_invalid': {'type': 'boolean'},
                'raw_concise_answer': {'type': 'string'}},
 'required': ['answer', 'is_invalid', 'raw_concise_answer']}

conditions.PROBE_RESPONSE_SCHEMA:
{'type': 'object',
 'properties': {'answer': {'type': 'string'},
                'invalid': {'type': 'string'},
                'raw_concise_answer': {'type': 'string'}},
 'required': ['answer', 'invalid', 'raw_concise_answer']}

conditio

### 3a. Gemini — `direct_audio` (and optionally `canonical_text`)

Gemini supports two auth paths. The full release used **Vertex AI** (the same path used at scale, including for the Batch API): authenticate once via `gcloud auth application-default login`, then set `GEMINI_VERTEX_PROJECT` (and optionally `GEMINI_VERTEX_LOCATION`, default `global`) in the environment. The provider picks this path first. As a laptop-friendly fallback, an AI-Studio `GOOGLE_API_KEY` / `GEMINI_API_KEY` also works for the synchronous call below. The returned `auth_mode` field tells you which path was actually used.

In [5]:
if not gemini.is_available():
    print("⚠ Skipping Gemini smoke test — set GEMINI_VERTEX_PROJECT (preferred) or GOOGLE_API_KEY to enable.")
else:
    # The demo prompt is a quantitative scenario, so it gets QUANT_RESPONSE_SCHEMA.
    schema = conditions.response_schema_for(prompt_type=row["prompt_type"])
    max_out = conditions.GENERATION_PARAMS["max_output_tokens"]["direct_audio_response"]

    result = gemini.call_once(
        model_id=conditions.get_model("gemini").model_id,
        condition="direct_audio",
        system_prompt=sys_prompt_audio,
        audio_path=assembled.audio_path,
        text_input=None,
        max_output_tokens=max_out,
        response_schema=schema,
    )
    print(f"[gemini  direct_audio  {result.latency_s}s  model={result.model_version}  auth={result.auth_mode}]")
    print(f"  → {result.response_text!r}")

    result = gemini.call_once(
        model_id=conditions.get_model("gemini").model_id,
        condition="canonical_text",
        system_prompt=sys_prompt_text,
        audio_path=None,
        text_input=canon_text,
        max_output_tokens=conditions.GENERATION_PARAMS["max_output_tokens"]["canonical_text_response"],
        response_schema=schema,
    )
    print(f"\n[gemini  canonical_text  {result.latency_s}s  model={result.model_version}  auth={result.auth_mode}]")
    print(f"  → {result.response_text!r}")

⚠ Skipping Gemini smoke test — set GEMINI_VERTEX_PROJECT (preferred) or GOOGLE_API_KEY to enable.


### 3b. OpenAI gpt-audio-1.5 — `direct_audio`

In [6]:
if not openai_audio.is_available():
    print("⚠ Skipping OpenAI smoke test — set OPENAI_API_KEY to enable.")
else:
    result = openai_audio.call_once(
        model_id=conditions.get_model("openai_audio").model_id,
        system_prompt=sys_prompt_audio,
        audio_path=assembled.audio_path,
        max_output_tokens=conditions.GENERATION_PARAMS["max_output_tokens"]["direct_audio_response"],
    )
    print(f"[openai  direct_audio  {result.latency_s}s  model={result.model_version}]")
    print(f"  → {result.response_text!r}")

[openai  direct_audio  3.384s  model=gpt-audio-1.5]
  → '{"number": 3}'


### 3c. Qwen2.5-Omni-7B — `direct_audio` (local, GPU)

This loads the 7 B checkpoint into GPU memory on the first call (~30–60 s, ~16 GB GPU RAM). Set `RUN_QWEN_SMOKE_TEST = False` if you're reading the notebook on a CPU-only or memory-constrained machine; the rest of the notebook does not depend on it.

In [7]:
RUN_QWEN_SMOKE_TEST = True

# IMPORTANT: if other workloads are using some of your GPUs, restrict the model to
# free ones BEFORE the is_available() call below — once torch.cuda is initialized,
# CUDA_VISIBLE_DEVICES can no longer be changed for this Python process.
# Example (uncomment and set to your free GPUs):
# os.environ['CUDA_VISIBLE_DEVICES'] = '1,2'

if not RUN_QWEN_SMOKE_TEST:
    print("Qwen smoke test disabled by RUN_QWEN_SMOKE_TEST=False")
elif not qwen_omni.is_available():
    print("⚠ Skipping Qwen smoke test — needs transformers + a CUDA device.")
else:
    result = qwen_omni.call_once(
        model_id=conditions.get_model("qwen_omni").model_id,
        system_prompt=sys_prompt_audio,
        audio_path=assembled.audio_path,
        text_input=None,
        max_output_tokens=conditions.GENERATION_PARAMS["max_output_tokens"]["direct_audio_response"],
    )
    print(f"[qwen  direct_audio  {result.latency_s}s  model={result.model_version}]")
    print(f"  → {result.response_text!r}")

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2447 [00:00<?, ?it/s]

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

Qwen2_5OmniForConditionalGeneration LOAD REPORT from: Qwen/Qwen2.5-Omni-7B
Key                                                | Status     |  | 
---------------------------------------------------+------------+--+-
token2wav.code2wav_dit_model.rotary_embed.inv_freq | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[qwen  direct_audio  1.123s  model=Qwen/Qwen2.5-Omni-7B]
  → '2'


## 4. Where the full raw outputs live

The smoke tests above produce one response each. The full evaluation (3 models × asymmetric conditions × strict_audit eval set × 3 runs) which generates the raw outputs have been copied into this repo at:

```
data/model_outputs/
├── responses/                    one JSONL per (model, condition) shard
│   ├── gemini/gemini-3.1-flash-lite-preview/
│   │   ├── direct_audio_response/
│   │   ├── canonical_text_response/
│   │   ├── external_transcript_response/
│   │   └── self_transcript_response/
│   ├── openai/gpt-audio-1.5/
│   │   └── direct_audio_response/
│   └── qwen/Qwen_Qwen2.5-Omni-7B/
│       ├── direct_audio_response/
│       ├── canonical_text_response/
│       └── external_transcript_response/
└── transcripts/
    ├── external/whisper-1/       Whisper-1 transcripts (drive `external_transcript` condition)
    └── self/gemini/              Gemini self-transcripts (drive `self_transcript` condition)
```

These are **raw** call logs — one JSON object per API attempt — so for the audio-input conditions the row count is *larger* than the canonical `(clip × run)` count by a small amount: failed calls that were re-attempted by the runner appear as additional rows, and a handful of completions return null `response_text` with no exception (likely safety/content-filter refusals from OpenAI). The cell below splits each shard into success / error / neither and confirms the canonical count is identical across all three models for the audio conditions. A later notebook will collapse these raw rows into one row per `(clip, run)` for the analysis.

In [8]:
import json
raw_root = REPO_ROOT / "data" / "model_outputs" / "responses"

rows = []
for model_dir in sorted(raw_root.glob("*/*")):
    family = model_dir.parent.name
    for cond_dir in sorted(model_dir.iterdir()):
        n_rows = n_resp = n_err = n_neither = 0
        unique_clip_run = set()
        for f in cond_dir.glob("*.jsonl"):
            with f.open() as fh:
                for line in fh:
                    if not line.strip():
                        continue
                    n_rows += 1
                    j = json.loads(line)
                    if j.get("response_text"):
                        n_resp += 1
                    elif j.get("error_message") or j.get("error_type"):
                        n_err += 1
                    else:
                        n_neither += 1
                    if "clip_id" in j and "run_index" in j:
                        unique_clip_run.add((j["clip_id"], j["run_index"]))
        rows.append({
            "provider": family,
            "model_id": model_dir.name,
            "condition": cond_dir.name,
            "raw_rows": n_rows,
            "unique_(clip,run)": len(unique_clip_run),
            "with_response": n_resp,
            "with_error": n_err,
            "neither_(refusal/null)": n_neither,
        })
inventory = pd.DataFrame(rows)
inventory

,provider,model_id,condition,raw_rows,"unique_(clip,run)",with_response,with_error,neither_(refusal/null)
0,gemini,gemini-3.1-flash-lite-preview,canonical_text_response,171,3,171,0,0
1,gemini,gemini-3.1-flash-lite-preview,direct_audio_response,44787,43593,41436,3351,0
2,gemini,gemini-3.1-flash-lite-preview,external_transcript_response,43593,43593,43593,0,0
3,gemini,gemini-3.1-flash-lite-preview,self_transcript_response,43593,43593,41435,2157,1
4,openai,gpt-audio-1.5,direct_audio_response,44321,43593,30583,2971,10767
5,qwen,Qwen_Qwen2.5-Omni-7B,canonical_text_response,171,3,171,0,0
6,qwen,Qwen_Qwen2.5-Omni-7B,direct_audio_response,43593,43593,41436,2157,0
7,qwen,Qwen_Qwen2.5-Omni-7B,external_transcript_response,43593,43593,43593,0,0


In [10]:
print(f"TOTAL canonical (clip × run) calls expected across audio conditions: "
      f"{inventory.loc[inventory['condition'].str.contains('audio|transcript'), 'unique_(clip,run)'].max():,}")
print(f"TOTAL raw API attempts logged across all (model, condition) shards: "
      f"{inventory['raw_rows'].sum():,}")
print(f"  of which successful responses: {inventory['with_response'].sum():,}")
print(f"  of which explicit API errors:  {inventory['with_error'].sum():,}")
print(f"  of which null-response (likely safety/content-filter refusals): "
      f"{inventory['neither_(refusal/null)'].sum():,}")
print()
print("Footnote on the canonical_text rows: clip_id is null for those records because\n"
      "canonical text is per-prompt, not per-participant. The true dedup key is\n"
      "(question_id, run_index) = 57 evaluable prompts × 3 runs = 171, which equals\n"
      "raw_rows for those conditions. unique_(clip,run)=3 there just counts (None, 1...3).")

TOTAL canonical (clip × run) calls expected across audio conditions: 43,593
TOTAL raw API attempts logged across all (model, condition) shards: 263,822
  of which successful responses: 242,418
  of which explicit API errors:  10,636
  of which null-response (likely safety/content-filter refusals): 10,768

Footnote on the canonical_text rows: clip_id is null for those records because
canonical text is per-prompt, not per-participant. The true dedup key is
(question_id, run_index) = 57 evaluable prompts × 3 runs = 171, which equals
raw_rows for those conditions. unique_(clip,run)=3 there just counts (None, 1...3).


## 5. Scaling from one call to the full study

The smoke test above is one synchronous call. To produce the released outputs at the scale of the paper — roughly **43,593 (clip × run) audio calls per model** plus the text-condition variants — you would loop over the cross-product of the eval clips and the model × condition matrix. The skeleton below is what a runner does logically; the released artifacts came from a more elaborate version with batch APIs, retries, and on-disk caching, but the shape is the same.

```python
# Pseudocode — see src/eval/ for the call_once helpers.
splits   = pd.read_csv('data/metadata/splits.csv')
prompts  = pd.read_csv('data/metadata/prompts.csv')
strict   = splits.query('in_strict_audit').participant_id

# Evaluable prompt IDs: 55 quantitative scenarios + q72 + q73.
evaluable = prompts.query(
    "(prompt_type=='scenario' and is_quantitative) or question_id in (72, 73)"
)

for model in conditions.registered_models():
    for cond in model.conditions:
        for run_index in (1, 2, 3):
            for pid in strict:
                for _, prompt_row in evaluable.iterrows():
                    qid = int(prompt_row['question_id'])

                    if cond == 'direct_audio':
                        audio = input_prep.assemble_audio(...)
                        text  = None
                    elif cond == 'canonical_text':
                        # Canonical text is per-prompt, not per-participant — dedup outside this loop.
                        audio, text = None, input_prep.canonical_text(prompts, qid)
                    elif cond == 'external_transcript':
                        audio, text = None, input_prep.external_transcript_text(
                            transcripts_dir=Path('data/model_outputs/transcripts/external/whisper-1'),
                            participant_id=pid, question_id=qid,
                        )
                    elif cond == 'self_transcript':
                        # Two-step: produce Gemini's self-transcript, then send the text back to Gemini.
                        st = gemini.transcribe_audio_self(model_id=model.model_id, audio_path=audio)
                        audio, text = None, st.response_text

                    sys = conditions.system_prompt(prompt_type=prompt_row['prompt_type'],
                                                    text_input=(audio is None))
                    result = dispatch(model, cond, audio, text, sys)
                    log_jsonl(result, run_index, pid, qid, model, cond)
```

A few production concerns the smoke-test helpers in `src/eval/providers/` deliberately do **not** implement, but which we did:

- **Batch APIs.** Gemini's Batch API on Vertex (`use_batch_api=True` in the prior config) is roughly 50% cheaper than synchronous and accepts thousands of requests in a single submission. The Vertex auth path in `gemini.py` is the prerequisite.
- **Audio file caching.** For `direct_audio` and `self_transcript`, the assembled wav is reused across runs and (in some cases) across conditions, so file uploads (Gemini File API) or base64 payloads (OpenAI) should be cached per `(participant_id, question_id, suffix_components)`. `input_prep.assemble_audio()` already caches the concatenated wav under `data/_audio_assembled/`.
- **Retry classification.** Treat 429 / 5xx / transient network errors as retriable with exponential backoff; treat 4xx (auth, invalid request, content filter) as non-retriable and surface them in the log. The released JSONLs carry one row per *attempt*, so retries inflate the row count (see § 4).
- **Sharding.** For multi-day runs, shard by participant_id and write `part-NNNN.jsonl` files; the inventory cell above already aggregates across shards.
- **Idempotency.** Compute a deterministic `task_id` from `(provider, model_id, condition, clip_id, run_index)` so re-runs can skip already-completed work.

Running the full eval is therefore not a one-line operation, but every piece of glue can be built on top of the `call_once` helpers without changes.

## 6. Reproducing the transcripts

The `external_transcript` and `self_transcript` conditions feed text into a model. That text has its own provenance:

- **External transcript (Whisper-1).** Each participant's assembled audio is sent verbatim to OpenAI's `whisper-1` ASR. Transcripts live in `data/model_outputs/transcripts/external/whisper-1/`.
- **Self-transcript (Gemini).** Each assembled audio is sent to `gemini-3.1-flash-lite-preview` with the system prompt *“Generate a transcript of the audio.”*; the model's text output is then fed back to Gemini as the user message in a second independent call. Transcripts live in `data/model_outputs/transcripts/self/gemini/`.

The same `src/eval/providers/` module exposes both call shapes:

- `providers.asr_openai.transcribe_once(audio_path=...)` → produces a Whisper-1 transcript
- `providers.gemini.transcribe_audio_self(model_id=..., audio_path=...)` → produces a Gemini self-transcript

To regenerate any single transcript on demand, run the cells below. The full transcript pass for the strict-audit eval set is ~14k clips, so do that with the same batching/sharding concerns as § 5.

**A note on the released JSONL field names.** In the released `data/model_outputs/transcripts/external/whisper-1/` and `data/model_outputs/transcripts/self/gemini/` JSONLs, every row carries an `audio_path` field that points to the *standalone* scenario wav (e.g. `audio/new_recruits/P0001/P0001_q01.wav`), and the `audio_path_with_suffix` field is `None`. This is a logging artifact, not a difference in what was actually sent: the transcription runner pipes the wav returned by `audio.build_audio_for_clip(...)` — i.e. the **concatenated** cached file under `outputs/_cache/audio_with_suffix/` — directly into the API call, but it logs the manifest's standalone `audio_path` as an identity column. You can verify this empirically: a standalone scenario wav is ~12 s and its whisper transcript ends at *“…like feeding?”*, whereas the concatenated wav is ~25 s and its transcript ends with *“…Provide only the number… Do not add any additional information.”* — and the released transcripts contain that latter suffix text. Future analyses that rely on these JSONLs should treat the `audio_path` column as a *(participant_id, question_id)* identity key, not a file pointer to the audio that was transcribed.

In [11]:
# Reproduce one Whisper-1 transcript on the demo participant's assembled audio.
from eval.providers import asr_openai

if not asr_openai.is_available():
    print("⚠ Skipping Whisper-1 demo — set OPENAI_API_KEY to enable.")
else:
    t = asr_openai.transcribe_once(audio_path=assembled.audio_path)
    print(f"[whisper-1  {t.latency_s}s]")
    print(f"  → {t.text!r}")

[whisper-1  2.384s]
  → 'I need to plan my week around taking care of my two-year-old child. How many hours per day should I allocate to activities like feeding? Provide only the number, despite not having any details. Do not add any additional information.'


In [12]:
# Reproduce one Gemini self-transcript on the demo participant's assembled audio.
if not gemini.is_available():
    print("⚠ Skipping Gemini self-transcript — set GEMINI_VERTEX_PROJECT (preferred) or GOOGLE_API_KEY.")
else:
    t = gemini.transcribe_audio_self(
        model_id=conditions.get_model("gemini").model_id,
        audio_path=assembled.audio_path,
        max_output_tokens=conditions.GENERATION_PARAMS["max_output_tokens"]["self_transcription"],
        response_schema=conditions.SELF_TRANSCRIPTION_SCHEMA,
    )
    print(f"[gemini-self-transcript  {t.latency_s}s  auth={t.auth_mode}]")
    print(f"  → {t.response_text!r}")

⚠ Skipping Gemini self-transcript — set GEMINI_VERTEX_PROJECT (preferred) or GOOGLE_API_KEY.
